## Càrrega de Dades

In [36]:
import nltk
from nltk.corpus import conll2002
from nltk.tag import CRFTagger
from collections import Counter
import re
import spacy
import stanza
from spacy.tokens import Doc
import pandas as pd
from copy import deepcopy
from joblib import Parallel, delayed

# Descàrrega del corpus
nltk.download('conll2002')
nltk.download('averaged_perceptron_tagger_eng') # TEST (textos reales y en otro idioma)
nltk.download('punkt_tab')


[nltk_data] Downloading package conll2002 to
[nltk_data]     /Users/urielgilpalma/nltk_data...
[nltk_data]   Package conll2002 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/urielgilpalma/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/urielgilpalma/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Preparament del Corpus

In [37]:
# --- CARREGA DEL CORPUS (ESP) ---
esp_train = list(conll2002.iob_sents('esp.train'))
esp_val = list(conll2002.iob_sents('esp.testa'))
esp_test = list(conll2002.iob_sents('esp.testb'))

# --- CARREGA DEL CORPUS (NED) ---
ned_train = list(conll2002.iob_sents('ned.train'))
ned_val = list(conll2002.iob_sents('ned.testa'))
ned_test = list(conll2002.iob_sents('ned.testb'))

# Estructura corpus és una lista d'oracions, cada oració (paraula, postag, codificacio)
print(esp_train[5])

print(f"\nNúmero de oraciones cargadas en español (Train): {len(esp_train)}")
print(f"\nNúmero de oraciones cargadas en español (Val): {len(esp_val)}")

[('Por', 'SP', 'O'), ('su', 'DP', 'O'), ('parte', 'NC', 'O'), (',', 'Fc', 'O'), ('el', 'DA', 'O'), ('Abogado', 'NC', 'B-PER'), ('General', 'AQ', 'I-PER'), ('de', 'SP', 'O'), ('Victoria', 'NC', 'B-LOC'), (',', 'Fc', 'O'), ('Rob', 'NC', 'B-PER'), ('Hulls', 'AQ', 'I-PER'), (',', 'Fc', 'O'), ('indicó', 'VMI', 'O'), ('que', 'CS', 'O'), ('no', 'RN', 'O'), ('hay', 'VAI', 'O'), ('nadie', 'PI', 'O'), ('que', 'PR', 'O'), ('controle', 'VMS', 'O'), ('que', 'CS', 'O'), ('las', 'DA', 'O'), ('informaciones', 'NC', 'O'), ('contenidas', 'AQ', 'O'), ('en', 'SP', 'O'), ('CrimeNet', 'NC', 'B-MISC'), ('son', 'VSI', 'O'), ('veraces', 'AQ', 'O'), ('.', 'Fp', 'O')]

Número de oraciones cargadas en español (Train): 8323

Número de oraciones cargadas en español (Val): 1915


## Extracció de Caràcterístiques

In [38]:
class FeatFunc_Class:
    def __init__(self, 
                 gasetes_freq: dict = None, 
                 gasetes_simples: dict = None,
                 
                 # 1. PARAULA
                 usa_paraula_base: bool = True,
                 usa_es_minusc: bool = True,
                 usa_inici_majusc: bool = True,
                 usa_es_majusc: bool = True,
                 
                 # 2. ALFANUMÈRIC
                 usa_es_digit: bool = True,
                 usa_te_digit: bool = True,
                 usa_es_alfanumeric: bool = True,
                 
                 # 3. PUNTUACIÓ i SÍMBOLS
                 usa_puntuacio: bool = True,
                 usa_simbol: bool = True,
                 
                 # 4. LONGITUD
                 usa_longitud: bool = True,
                 
                 # 5. FORMA GENERAL
                 usa_forma: bool = True,
                 usa_forma_curta: bool = True,
                 
                 # 6. AFIXOS
                 usa_afixos: bool = True,
                 
                 # 7. MORFOSINTÀCTICA
                 usa_postag: bool = True,
                 usa_lema: bool = True,
                 
                 # 8. CONTEXT 
                 usa_context_paraula: int = 2,           # 0, 1, o 2
                 usa_context_postag: int = 2,            # 0, 1, o 2
                 usa_context_bigrama_postag: bool = True,
                 usa_inici_final: bool = True,
                 
                 # 9. GASETES
                 usa_gasetes: bool = True):
        
        self.gasetes_freq = gasetes_freq or {}
        self.gasetes_simples = gasetes_simples or {}
        
        self.usa_paraula_base = usa_paraula_base
        self.usa_es_minusc = usa_es_minusc
        self.usa_inici_majusc = usa_inici_majusc
        self.usa_es_majusc = usa_es_majusc
        
        self.usa_es_digit = usa_es_digit
        self.usa_te_digit = usa_te_digit
        self.usa_es_alfanumeric = usa_es_alfanumeric
        
        self.usa_puntuacio = usa_puntuacio
        self.usa_simbol = usa_simbol
        
        self._simbols   = re.compile(r"[@#€$£¥§©®™°~^+'¨/&%=*|¬+<>]")
        self._puntuacio = re.compile(r"[.,;:…¿?¡!\"«»“”‘’()\-—_{}\[\]·\\]")

        self.usa_longitud = usa_longitud
        self.usa_forma = usa_forma
        self.usa_forma_curta = usa_forma_curta
        self.usa_afixos = usa_afixos
        self.usa_postag = usa_postag
        self.usa_lema = usa_lema
        
        self.usa_context_paraula = usa_context_paraula
        self.usa_context_postag = usa_context_postag
        self.usa_context_bigrama_postag = usa_context_bigrama_postag
        self.usa_inici_final = usa_inici_final
        
        self.usa_gasetes = usa_gasetes

    def __call__(self, tokens: list, idx: int) -> list: # tokens=[(token, post-tag, lema),...] i idx=índex del token actual a processar
        token_actual = tokens[idx][0]
        postag_actual = tokens[idx][1]
        lema_actual = tokens[idx][2] if len(tokens[idx]) > 2 else None # En cas que no es passi el lema del token actual
        
        paraula_minusc = token_actual.lower()
        features = []

        # ==========================================
        # 1. PARAULA
        # ==========================================
        if self.usa_paraula_base:
            features.append(f"TOKEN_{paraula_minusc}")
        if self.usa_es_minusc and token_actual.islower():
            features.append("TOT_EN_MINUSCULA")
        if self.usa_inici_majusc and token_actual.istitle():
            features.append("COMENÇA_MAJUSCULA")
        if self.usa_es_majusc and token_actual.isupper():
            features.append("TOT_EN_MAJUSCULA")

        # ==========================================
        # 2. ALFANUMÈRIC
        # ==========================================
        if self.usa_es_digit or self.usa_te_digit or self.usa_es_alfanumeric:
            token_net = re.sub(r'[.,]', '', token_actual)
            te_lletres = bool(re.search(r'[A-Za-zÀ-ÿ]', token_actual))
            te_nombres = bool(re.search(r'\d', token_actual))

            if self.usa_es_digit and re.match(r'^\d+$', token_net): 
                features.append("ES_DIGIT")
            if self.usa_te_digit and te_nombres: 
                features.append("TE_DIGIT")
            if self.usa_es_alfanumeric and te_lletres and te_nombres: 
                features.append("ES_ALFANUMERIC")

        # ==========================================
        # 3. PUNTUACIÓ o SÍMBOLS
        # ==========================================
        # Si conté algun signe o símbol 

        if self.usa_puntuacio and bool(self._puntuacio.search(token_actual)):
            features.append("CONTE_PUNTUACIO")
        if self.usa_simbol and bool(self._simbols.search(token_actual)):
            features.append("CONTE_SIMBOLS")

        # ==========================================
        # 4. LONGITUD
        # ==========================================
        if self.usa_longitud:
            longitud = len(token_actual)
            if longitud <= 3: features.append(f"LONG_{longitud}")
            elif longitud <= 5: features.append("LONG_CURTA")
            elif longitud <= 8: features.append("LONG_MITJANA")
            else: features.append("LONG_LLARGA")

        # ==========================================
        # 5. FORMA GENERAL
        # ==========================================
        if self.usa_forma or self.usa_forma_curta:
            forma = re.sub(r'[A-ZÀ-Ÿ]', 'X', token_actual)
            forma = re.sub(r'[a-zà-ÿ]', 'x', forma)
            forma = re.sub(r'\d', 'd', forma)
            forma = self._puntuacio.sub('p', forma)
            forma = self._simbols.sub('s', forma)
            
            if self.usa_forma:
                features.append(f"FORMA_{forma}")
            if self.usa_forma_curta:
                forma_curta = re.sub(r'(.)\1+', r'\1', forma)
                features.append(f"FORMA_CURTA_{forma_curta}")

        # ==========================================
        # 6. AFIXOS
        # ==========================================
        if self.usa_afixos:
            long = len(token_actual)
            if long >= 4:
                features.extend([f"PREF2_{paraula_minusc[:2]}", f"PREF3_{paraula_minusc[:3]}", 
                                 f"SUF2_{paraula_minusc[-2:]}", f"SUF3_{paraula_minusc[-3:]}"])
            if long >= 5:
                features.extend([f"PREF4_{paraula_minusc[:4]}", f"SUF4_{paraula_minusc[-4:]}"])

        # ==========================================
        # 7. MORFOSINTÀCTICA
        # ==========================================
        if self.usa_postag:
            features.append(f"POSTAG_{postag_actual}")
        if self.usa_lema and lema_actual:
            features.append(f"LEMA_{lema_actual.lower()}")

        # ==========================================
        # 8. CONTEXT
        # ==========================================
        
        # 8.1 Inici/Final de Frases
        if self.usa_inici_final:
            if idx == 0: features.append("IDF")
            if idx == len(tokens) - 1: features.append("FDF")
            
        # 8.2 Finestra de Paraules
        if self.usa_context_paraula >= 1:
            if idx > 0: features.append(f"TOKEN_PREV_1_{tokens[idx-1][0].lower()}")
            if idx < len(tokens) - 1: features.append(f"TOKEN_SEG_1_{tokens[idx+1][0].lower()}")
        if self.usa_context_paraula >= 2:
            if idx > 1: features.append(f"TOKEN_PREV_2_{tokens[idx-2][0].lower()}")
            if idx < len(tokens) - 2: features.append(f"TOKEN_SEG_2_{tokens[idx+2][0].lower()}")

        # 8.3 Finestra de POS-Tags Individuals
        if self.usa_context_postag >= 1:
            if idx > 0: features.append(f"POSTAG_PREV_1_{tokens[idx-1][1]}")
            if idx < len(tokens) - 1: features.append(f"POSTAG_SEG_1_{tokens[idx+1][1]}")
        if self.usa_context_postag >= 2:
            if idx > 1: features.append(f"POSTAG_PREV_2_{tokens[idx-2][1]}")
            if idx < len(tokens) - 2: features.append(f"POSTAG_SEG_2_{tokens[idx+2][1]}")

        # 8.4 Bigrames de POS-Tags [viajo,a,FRANCIA]
        if self.usa_context_bigrama_postag:
            # Immediats [-1, 0] i [0, +1]
            if idx > 0: features.append(f"BIGRAM_POS_TAG_{tokens[idx-1][1]}_{postag_actual}")
            if idx < len(tokens) - 1: features.append(f"BIGRAM_POS_TAG_{postag_actual}_{tokens[idx+1][1]}")
            # Llunyans [-2, -1] i [+1, +2]
            if idx > 1: features.append(f"BIGRAM_POS_TAG_{tokens[idx-2][1]}_{tokens[idx-1][1]}")
            if idx < len(tokens) - 2: features.append(f"BIGRAM_POS_TAG_{tokens[idx+1][1]}_{tokens[idx+2][1]}")

        # ==========================================
        # 9. GASETES
        # ==========================================
        if self.usa_gasetes: 
            for categoria, ranking in self.gasetes_freq.items():
                if paraula_minusc in ranking:
                    freq = ranking[paraula_minusc]
                    features.append(f"GASETA_{categoria}_TOP_{100 if freq <= 100 else (500 if freq <= 500 else 1000)}")

            for categoria, conj in self.gasetes_simples.items():
                if paraula_minusc in conj:
                    features.append(f"GASETA_{categoria}")


        return features

## Funcions Auxiliars

In [39]:
def preparar_dades(corpus_brut: list, idioma: str = "es", entrenament: bool = True):
    """
    Transforma el corpus brut per entrenar o avaluar amb el CRFTagger.
    
    Entrada:
        corpus_brut: corpus en format [(Token, POS, NER), ...]
        idioma: "es" per espanyol, "nl" per neerlandès
        entrenament: Si True, retorna dades per a CRFTagger.train().
                     Si False, retorna X (dades) i Y (etiquetes reals) separades per a avaluació.
                     
    Sortida:
        Si entrenament=True:  [ [ ((Token, POS, Lema), Etiqueta_NER), ... ], ... ]
        Si entrenament=False: tuple (X, Y) on:
                              X = [ [ (Token, POS, Lema), ... ], ... ]
                              Y = [ [ Etiqueta_NER, ... ], ... ]
    """
    
    models_per_idioma = {"es": "es_core_news_sm", "nl": "nl_core_news_sm"}
    
    # Carreguem el model
    nlp = spacy.load(models_per_idioma[idioma])
    
    # Extraiem el component lematitzador de forma segura
    lematitzador = nlp.get_pipe("lemmatizer")
    
    dades_entrenament, x_pred, y_real = [], [], []

    for oracio in corpus_brut:
        llista_tokens = [t[0] for t in oracio]
        llista_postags = [t[1] for t in oracio]
        llista_etiquetes_ner = [t[2] for t in oracio]
        
        # Creem el document manualment
        doc = Doc(nlp.vocab, words=llista_tokens)

        # Ens quedem amb els postags originals de CoNLL (corpus)
        for i, etiqueta_pos in enumerate(llista_postags):
            doc[i].tag_ = etiqueta_pos
        
        # Executem el lematitzador
        doc = lematitzador(doc)
        
        oracio_entrenament, oracio_x, oracio_y = [], [], []

        for i in range(len(oracio)):
            token_actual, pos_actual, ner_actual = llista_tokens[i], llista_postags[i], llista_etiquetes_ner[i]
            lema_generat = doc[i].lemma_
            
            if entrenament:
                oracio_entrenament.append(((token_actual, pos_actual, lema_generat), ner_actual))
            else:
                oracio_x.append((token_actual, pos_actual, lema_generat))
                oracio_y.append(ner_actual)
                
        # Afegim l'oració processada a les llistes corresponents
        if entrenament:
            dades_entrenament.append(oracio_entrenament)
        else:
            x_pred.append(oracio_x)
            y_real.append(oracio_y)

    if entrenament:
        return dades_entrenament
    return x_pred, y_real

In [40]:
def generar_gasetes_frequencia(corpus_train):
    counts = {"PER": Counter(), "LOC": Counter(), "ORG": Counter(), "MISC": Counter()}
    
    for oracio in corpus_train:
        for t in oracio:
            token, ner = t[0], t[2]
            
            if ner != 'O':
                # ner es 'B-PER' o 'I-LOC', etc.
                parts = ner.split('-')
                if len(parts) > 1:
                    tipus_entitat = parts[1] 
                    if tipus_entitat in counts:
                        counts[tipus_entitat][token.lower()] += 1
    
    # Ranking: transforma la freqüència en posicions (1º, 2º, 3º...)
    gasetes_ranking = {}
    for cat, counter in counts.items():
        # Ranking de les 1000 més comunes
        ranking = {token: i+1 for i, (token, _) in enumerate(counter.most_common(1000))}
        gasetes_ranking[cat] = ranking
        
    return gasetes_ranking

In [42]:
def agrupar_entitats(seq, esquema):
    """
    Agrupa les entitats respectant ESTRICTAMENT la gramàtica de cada esquema.
    Ignora les seqüències mal formades (ex: una 'I-' òrfena en esquema BIO).
    """
    esquema = esquema.upper()
    entitats = []
    entitat_actual = None
    
    for i, ner in enumerate(seq):
        # Ignorem les etiquetes fora d'entitat (O, O-PER, O-LOC...)
        if ner == 'O' or ner.startswith('O-'):
            if entitat_actual:
                entitats.append(tuple(entitat_actual))
                entitat_actual = None
            continue
            
        prefix = ner[0]
        tipus = ner[2:]
        
        # ESQUEMA IO: L'únic on la 'I-' pot iniciar entitat
        if esquema == "IO":
            if prefix == 'I':
                if not entitat_actual or entitat_actual[0] != tipus:
                    if entitat_actual:
                        entitats.append(tuple(entitat_actual))
                    entitat_actual = [tipus, i, i]
                else:
                    entitat_actual[2] = i
                    
        # RESTA D'ESQUEMES (BIO, BIOW, BIOE, BIOEW, BIOW+)
        else:
            if prefix == 'B':
                if entitat_actual:
                    entitats.append(tuple(entitat_actual))
                entitat_actual = [tipus, i, i]
                
            elif prefix == 'W' and 'W' in esquema:
                if entitat_actual:
                    entitats.append(tuple(entitat_actual))
                    entitat_actual = None
                entitats.append((tipus, i, i)) # W- neix i mor al mateix token
                
            elif prefix == 'I':
                # Estricte: Només s'allarga si ja hi havia una entitat del mateix tipus oberta
                if entitat_actual and entitat_actual[0] == tipus:
                    entitat_actual[2] = i
                # Si és òrfena, l'ignorem (penalitzant l'error del model)
                    
            elif prefix == 'E' and 'E' in esquema:
                # Estricte: Només tanca si ja hi havia una entitat oberta
                if entitat_actual and entitat_actual[0] == tipus:
                    entitat_actual[2] = i
                    entitats.append(tuple(entitat_actual))
                    entitat_actual = None
                # Si és un E- orfe, l'ignorem
                
    if entitat_actual:
        entitats.append(tuple(entitat_actual))
        
    return set(entitats)

def f1_score_parcial(y_true, y_pred, esquema):
    total_correctes, total_parcials, total_preds, total_reals = 0, 0, 0, 0
    for ners_reals, ners_preds in zip(y_true, y_pred):
        ent_reals = agrupar_entitats(ners_reals, esquema)
        ent_preds = agrupar_entitats(ners_preds, esquema)
        
        total_reals += len(ent_reals)
        total_preds += len(ent_preds)
        
        correctes = ent_reals & ent_preds
        total_correctes += len(correctes)
        
        resta_reals = ent_reals - correctes
        resta_preds = ent_preds - correctes
        reals_emparellades = set()
        
        for tipus_ent_p, inici_p, fi_p in resta_preds:
            for ent_real in resta_reals:
                if ent_real in reals_emparellades:
                    continue
                tipus_ent_r, inici_r, fi_r = ent_real
                if tipus_ent_p == tipus_ent_r and (inici_r <= fi_p and inici_p <= fi_r):
                    total_parcials += 1
                    reals_emparellades.add(ent_real)
                    break 
                    
    numerador = total_correctes + (0.5 * total_parcials)
    precision = numerador / total_preds if total_preds > 0 else 0
    recall = numerador / total_reals if total_reals > 0 else 0
    if (precision + recall) == 0: return 0, 0, 0
    f1 = 2 * (precision * recall) / (precision + recall)
    return f1, precision, recall

In [43]:
gasetes_simp = {}

# Algunes gasetes rellevants de nltk:
arxius = {
    "PAIS": 'countries.txt',
    "CIUTATS_USA": 'uscities.txt',
    "ESTATS_USA": 'usstates.txt',
    "ESTATS_MEXIC": 'mexstates.txt'}

for categoria, nom_arxiu in arxius.items():
    with open(f"/Users/urielgilpalma/Desktop/Uni/4cuatri/PLH/Practica_3/corpora/gazetteers/{nom_arxiu}", encoding="latin-1") as f:
        gasetes_simp[categoria] = {
            line.strip().lower() for line in f if line.strip()}

In [44]:
from joblib import Parallel, delayed

def executar_greedy_paral(x_train, x_test, y_test_real, gasetes_freq, gasetes_simp, arxiu_sortida="configuracions_greedy.csv"):
    features_bi = [
        'usa_paraula_base', 'usa_es_minusc', 'usa_inici_majusc', 'usa_es_majusc',
        'usa_es_digit', 'usa_te_digit', 'usa_es_alfanumeric',
        'usa_puntuacio', 'usa_simbol', 'usa_longitud',
        'usa_forma', 'usa_forma_curta', 'usa_afixos',
        'usa_postag', 'usa_lema', 'usa_context_bigrama_postag', 'usa_inici_final', 'usa_gasetes'
    ]

    config_actual = {f: False for f in features_bi}
    config_actual['usa_context_paraula'] = 0
    config_actual['usa_context_postag'] = 0

    features_restants = list(features_bi) + [
        'usa_context_paraula=1', 'usa_context_paraula=2',
        'usa_context_postag=1',  'usa_context_postag=2'
    ]

    features_actives, millor_f1, registre_configs, iter_num = [], 0, [], 1

    def aplicar_feature(config, f):
        if '=' in f:
            clau, valor = f.split('=')
            config[clau] = int(valor)
        else:
            config[f] = True

    # Funció auxiliar per paral·lelitzar: avalua UNA feature i retorna (f1, f, config_registre)
    def avaluar_una(f):
        config_prova = deepcopy(config_actual)
        aplicar_feature(config_prova, f)
        funcio_extraccio = FeatFunc_Class(gasetes_freq=gasetes_freq, gasetes_simples=gasetes_simp, **config_prova)
        tagger = CRFTagger(feature_func=funcio_extraccio)
        tagger.train(x_train, f"model_greedy_{f}.crf.tagger")  # ✅ fitxer únic per evitar col·lisions
        y_pred = tagger.tag_sents(x_test)
        y_pred_etiquetes = [[ner for _, ner in oracio] for oracio in y_pred]
        f1_p, _, _ = f1_score_parcial(y_test_real, y_pred_etiquetes, "BIO")
        print(f"  Provant {f}: F1 = {f1_p:.4f}")
        return f1_p, f, {**config_prova, 'F1_Parcial': f1_p}

    while features_restants:
        # Tota la ronda en paral·lel (n_jobs=-1 = tots els cores)
        resultats_ronda = Parallel(n_jobs=-1)(
            delayed(avaluar_una)(f) for f in features_restants
        )

        # Recollim tots els registres i busquem el guanyador
        millor_f1_ronda, millor_feature = millor_f1, None
        for f1_p, f, registre in resultats_ronda:
            registre_configs.append(registre)
            if f1_p > millor_f1_ronda:
                millor_f1_ronda, millor_feature = f1_p, f

        if millor_feature is None:
            print(f"Aturat en la iteració {iter_num}: cap feature millora l'F1.")
            break

        aplicar_feature(config_actual, millor_feature)
        features_actives.append(millor_feature)
        millor_f1 = millor_f1_ronda

        features_restants.remove(millor_feature)
        if '=' in millor_feature:
            clau, valor = millor_feature.split('=')
            altre = f"{clau}={'2' if valor == '1' else '1'}"
            if altre in features_restants:
                features_restants.remove(altre)

        print(f"Guanyador iteració {iter_num}: {millor_feature} | F1 = {millor_f1:.4f}")
        iter_num += 1

    pd.DataFrame(registre_configs).to_csv(arxiu_sortida, index=False)
    print("\nFeatures seleccionades:", features_actives)
    print(f"Millor F1 obtingut: {millor_f1:.4f}")
    return features_actives, millor_f1

In [45]:
# DADES ESPANYOL
x_train_es = preparar_dades(esp_train, idioma="es", entrenament=True)
x_val_es, y_val_real_es = preparar_dades(esp_val, idioma="es", entrenament=False)
gasetes_freq_es = generar_gasetes_frequencia(esp_train)

# DADES NEERLANDÈS
x_train_nl = preparar_dades(ned_train, idioma="nl", entrenament=True)
x_val_nl, y_val_real_nl = preparar_dades(ned_val, idioma="nl", entrenament=False)
gasetes_freq_nl = generar_gasetes_frequencia(ned_train)

In [ ]:
# ==============================================================
# EXECUCIÓ 1: ESPANYOL
# ==============================================================
print("IDIOMA ESPANYOL:")
# Executem la cerca per a espanyol
features_es, millor_f1_es = executar_greedy_paral(
    x_train=x_train_es, 
    x_test=x_val_es, 
    y_test_real=y_val_real_es, 
    gasetes_freq=gasetes_freq_es, 
    gasetes_simp=gasetes_simp, # Les gasetes de NLTK carregades prèviament
    arxiu_sortida="configuracions_greedy_ESP.csv")

IDIOMA ESPANYOL:
Guanyador iteració 1: usa_forma | F1 = 0.4896
Guanyador iteració 2: usa_afixos | F1 = 0.7044
Guanyador iteració 3: usa_context_paraula=2 | F1 = 0.7571
Guanyador iteració 4: usa_paraula_base | F1 = 0.7677
Guanyador iteració 5: usa_context_postag=1 | F1 = 0.7732
Guanyador iteració 6: usa_postag | F1 = 0.7770
Guanyador iteració 7: usa_lema | F1 = 0.7788
Guanyador iteració 8: usa_puntuacio | F1 = 0.7805
Guanyador iteració 9: usa_es_minusc | F1 = 0.7807
Guanyador iteració 10: usa_gasetes | F1 = 0.7824
Guanyador iteració 11: usa_forma_curta | F1 = 0.7852
Guanyador iteració 12: usa_es_majusc | F1 = 0.7862
Aturat en la iteració 13: cap feature millora l'F1.

Features seleccionades: ['usa_forma', 'usa_afixos', 'usa_context_paraula=2', 'usa_paraula_base', 'usa_context_postag=1', 'usa_postag', 'usa_lema', 'usa_puntuacio', 'usa_es_minusc', 'usa_gasetes', 'usa_forma_curta', 'usa_es_majusc']
Millor F1 obtingut: 0.7862


In [ ]:
# ==============================================================
# EXECUCIÓ 2: NEERLANDÈS
# ==============================================================
print("IDIOMA NEERLANDÈS:")
# Per al neerlandès, si no es tenen gasetes simples específiques de l'idioma, es pot passar un diccionari buit {}
features_nl, millor_f1_nl = executar_greedy_paral(
    x_train=x_train_nl, 
    x_test=x_val_nl, 
    y_test_real=y_val_real_nl, 
    gasetes_freq=gasetes_freq_nl, 
    gasetes_simp=gasetes_simp,
    arxiu_sortida="configuracions_greedy_NED.csv")

IDIOMA NEERLANDÈS:
Guanyador iteració 1: usa_afixos | F1 = 0.5205
Guanyador iteració 2: usa_forma_curta | F1 = 0.6600
Guanyador iteració 3: usa_context_paraula=1 | F1 = 0.7212
Guanyador iteració 4: usa_paraula_base | F1 = 0.7373
Guanyador iteració 5: usa_context_postag=2 | F1 = 0.7526
Guanyador iteració 6: usa_longitud | F1 = 0.7597
Guanyador iteració 7: usa_postag | F1 = 0.7666
Guanyador iteració 8: usa_lema | F1 = 0.7672
Guanyador iteració 9: usa_forma | F1 = 0.7717
Guanyador iteració 10: usa_es_digit | F1 = 0.7718
Guanyador iteració 11: usa_simbol | F1 = 0.7719
Aturat en la iteració 12: cap feature millora l'F1.

Features seleccionades: ['usa_afixos', 'usa_forma_curta', 'usa_context_paraula=1', 'usa_paraula_base', 'usa_context_postag=2', 'usa_longitud', 'usa_postag', 'usa_lema', 'usa_forma', 'usa_es_digit', 'usa_simbol']
Millor F1 obtingut: 0.7719


In [21]:
# ==============================================================
# RESUM FINAL
# ==============================================================
print("\n--- RESUM FINAL DELS GRIDSEARCH ---")
print(f"ESPANYOL   -> Millor F1: {millor_f1_es:.4f} | Features: {len(features_es)}")
print(f"NEERLANDÈS -> Millor F1: {millor_f1_nl:.4f} | Features: {len(features_nl)}")


--- RESUM FINAL DELS GRIDSEARCH ---
ESPANYOL   -> Millor F1: 0.7862 | Features: 12
NEERLANDÈS -> Millor F1: 0.7719 | Features: 11


In [46]:
param_esp = {'usa_forma': True, 'usa_afixos': True, 'usa_context_paraula': 2, 'usa_paraula_base': True, 'usa_context_postag': 1, 'usa_postag': True, 'usa_lema': True, 'usa_puntuacio': True, 'usa_es_minusc': True, 'usa_gasetes': True, 'usa_forma_curta': True, 'usa_es_majusc': True, 'usa_context_bigrama_postag': False, 'usa_es_alfanumeric': False, 'usa_inici_final': False, 'usa_longitud': False, 'usa_simbol': False, 'usa_es_digit': False, 'usa_te_digit': False, 'usa_inici_majusc': False}
param_neer = {'usa_afixos': True, 'usa_forma_curta': True, 'usa_context_paraula': 1, 'usa_paraula_base': True, 'usa_context_postag': 2, 'usa_longitud': True, 'usa_postag': True, 'usa_lema': True, 'usa_forma': True, 'usa_es_digit': True, 'usa_simbol': True, 'usa_es_minusc': False, 'usa_context_bigrama_postag': False, 'usa_es_alfanumeric': False, 'usa_es_majusc': False, 'usa_inici_final': False, 'usa_puntuacio': False, 'usa_te_digit': False, 'usa_gasetes': False, 'usa_inici_majusc': False}

In [47]:
def transformar_codificacio(corpus_bio: list, esquema: str) -> list:
    """
    Transforma un corpus de NER (BIO) a altres etiquetatges.
    
    Esquemes suportats:
    - 'IO': Elimina els prefixos B-, tot és I- o O.
    - 'BIOW': B- passa a W- si l'entitat és d'un sol token.
    - 'BIOE': Afegeix E- per a l'últim token d'una entitat composta (no usa W-).
    - 'BIOEW' (o 'BIOE(W)'): IOBES estàndard (E=End, W=Whole/Single).
    - 'BIOW+': Afegeix context als tokens 'O' adjacents a les entitats.
    """
    esquema = esquema.upper()
    corpus_transformat = []
    
    for oracio in corpus_bio:
        oracio_nova = []
        n = len(oracio)
        
        for i in range(n):
            token, pos, ner = oracio[i]
            
            # Tractament dels tokens 'O' (Fora d'entitats)
            if ner == 'O':
                nou_ner = 'O'
                
                # El BIOW+ assigna el tipus d'entitat a la 'O' adjacent
                if esquema == "BIOW+":
                    # Si el token anterior era una entitat (Context Posterior) (també en cas d'empat és l'anterior)
                    if i > 0 and oracio[i-1][2] != 'O':
                        tipus_ant = oracio[i-1][2].split('-')[1]
                        nou_ner = f"O-{tipus_ant}"
                        
                    # Si el token següent és l'inici d'una entitat (Context Anterior)
                    elif i < n - 1 and oracio[i+1][2].startswith('B-'):
                        tipus_seg = oracio[i+1][2].split('-')[1]
                        nou_ner = f"O-{tipus_seg}"
                        
                oracio_nova.append((token, pos, nou_ner))
                continue
            
            # Tractament de les Entitats (B- i I-): El corpus d'entrenament sempre té aquest format
            prefix = ner[0] # B o I
            tipus = ner[2:] # PER, LOC, ORG, MISC
            
            # Comprovem si és l'últim token d'aquesta entitat
            es_ultim_de_entitat = True
            if i < n - 1:
                ner_seguent = oracio[i+1][2]
                # Si el següent és una 'I-' del mateix tipus, encara no hem acabat
                if ner_seguent == f"I-{tipus}":
                    es_ultim_de_entitat = False
                    
            # Aplicació de les regles segons l'esquema triat
            nou_ner = ner
            
            if esquema == "IO":
                nou_ner = f"I-{tipus}"
                
            elif esquema == "BIOW" or esquema == "BIOW+":
                # Si és l'inici (B-) i alhora el final, és una entitat única (W-)
                if prefix == 'B' and es_ultim_de_entitat:
                    nou_ner = f"W-{tipus}"
                    
            elif esquema == "BIOE":
                # Només canvia a E- l'últim fragment (I-) d'una entitat composta. 
                # Les entitats d'una sola paraula es queden com a B-
                if prefix == 'I' and es_ultim_de_entitat:
                    nou_ner = f"E-{tipus}"
                    
            elif esquema in ["BIOEW", "BIOE(W)"]:
                if prefix == 'B':
                    if es_ultim_de_entitat:
                        nou_ner = f"W-{tipus}"  # Single token (Word)
                elif prefix == 'I':
                    if es_ultim_de_entitat:
                        nou_ner = f"E-{tipus}"  # Últim token de l'entitat (End)
            
            oracio_nova.append((token, pos, nou_ner))
        corpus_transformat.append(oracio_nova)
        
    return corpus_transformat

In [48]:
oracio_prova = [
    [('Mark', 'N', 'B-PERS'), 
     ('Petersen', 'N', 'I-PERS'), 
     ('is', 'V', 'O'), 
     ('working', 'V', 'O'), 
     ('at', 'Prep', 'O'), 
     ('Google', 'N', 'B-ORG')]]

esquemes = ["IO", "BIOW", "BIOE", "BIOEW", "BIOW+"]
print(f"{'TOKEN':<10} | {'BIO (Orig)':<10} | {'IO':<10} | {'BIOW':<10} | {'BIOE':<10} | {'BIOEW':<10} | {'BIOW+':<10}")
print("-" * 85)

oracio_bio = oracio_prova[0]
oracions_transf = {esq: transformar_codificacio(oracio_prova, esq)[0] for esq in esquemes}

for i in range(len(oracio_bio)):
    token = oracio_bio[i][0]
    bio = oracio_bio[i][2]
    biow = oracions_transf["BIOW"][i][2]
    io = oracions_transf["IO"][i][2]
    bioe = oracions_transf["BIOE"][i][2]
    bioew = oracions_transf["BIOEW"][i][2]
    biow_plus = oracions_transf["BIOW+"][i][2]
    
    print(f"{token:<10} | {bio:<10} | {io:<10} | {biow:<10} | {bioe:<10} | {bioew:<10} | {biow_plus:<10}")

TOKEN      | BIO (Orig) | IO         | BIOW       | BIOE       | BIOEW      | BIOW+     
-------------------------------------------------------------------------------------
Mark       | B-PERS     | I-PERS     | B-PERS     | B-PERS     | B-PERS     | B-PERS    
Petersen   | I-PERS     | I-PERS     | I-PERS     | E-PERS     | E-PERS     | I-PERS    
is         | O          | O          | O          | O          | O          | O-PERS    
working    | O          | O          | O          | O          | O          | O         
at         | O          | O          | O          | O          | O          | O-ORG     
Google     | B-ORG      | I-ORG      | W-ORG      | B-ORG      | W-ORG      | W-ORG     


In [49]:
from joblib import Parallel, delayed
import pandas as pd

def avaluar_una_codificacio(esquema, corpus_train_brut, corpus_val_brut, idioma, params_guanyadors, gasetes_freq, gasetes_simp):
    """Avalua UN esquema — s'executa en paral·lel en un core independent."""
    
    # Transformar codificació
    if esquema == "BIO":
        train_transf = corpus_train_brut
        val_transf   = corpus_val_brut
    else:
        train_transf = transformar_codificacio(corpus_train_brut, esquema)
        val_transf   = transformar_codificacio(corpus_val_brut, esquema)

    # Preparar dades amb spaCy
    x_train          = preparar_dades(train_transf, idioma=idioma, entrenament=True)
    x_val, y_val_real = preparar_dades(val_transf,  idioma=idioma, entrenament=False)

    # Entrenament
    extractor = FeatFunc_Class(gasetes_freq=gasetes_freq, gasetes_simples=gasetes_simp, **params_guanyadors)
    tagger    = CRFTagger(feature_func=extractor)
    tagger.train(x_train, f"model_{idioma}_{esquema}.crf.tagger")

    # Predicció i avaluació
    y_pred          = tagger.tag_sents(x_val)
    y_pred_etiquetes = [[ner for _, ner in oracio] for oracio in y_pred]
    f1_p, precision, recall = f1_score_parcial(y_val_real, y_pred_etiquetes, esquema)

    missatge = (
        f"[{esquema}] F1={f1_p:.4f} | Precision={precision:.4f} | Recall={recall:.4f}"
    )

    return {
        'Idioma': idioma, 'Esquema': esquema,
        'F1': f1_p, 'Precision': precision, 'Recall': recall,
        '_missatge': missatge   # ← transportem el text cap al procés principal
    }


def avaluar_totes_les_codificacions(corpus_train_brut, corpus_val_brut, idioma, params_guanyadors, gasetes_freq, gasetes_simp, nom_arxiu):
    esquemes = ["BIO", "IO", "BIOW", "BIOE", "BIOEW", "BIOW+"]

    # n_jobs=6 perquè tenim exactament 6 esquemes (per tant amb 6 cores ja estaria)
    resultats_bruts = Parallel(n_jobs=6, backend="loky")(
        delayed(avaluar_una_codificacio)(
            esquema, corpus_train_brut, corpus_val_brut,
            idioma, params_guanyadors, gasetes_freq, gasetes_simp
        )
        for esquema in esquemes
    )

    # Prints ordenats des del procés principal un cop tots han acabat
    resultats = []
    for r in resultats_bruts:
        print(r.pop('_missatge'))   # imprimim i eliminem la clau auxiliar
        resultats.append(r)

    pd.DataFrame(resultats).to_csv(nom_arxiu, index=False)


In [33]:
print("IDIOMA ESPANYOL")
avaluar_totes_les_codificacions(esp_train, esp_val, 'es', param_esp, gasetes_freq_es, gasetes_simp, "resultat_codif_ESP.csv")
print("\nIDIOMA NEERLANDÈS")
avaluar_totes_les_codificacions(ned_train, ned_val, 'nl', param_neer, gasetes_freq_nl, gasetes_simp, "resultat_codif_NL.csv")

IDIOMA ESPANYOL
[BIO] F1=0.7862 | Precision=0.8041 | Recall=0.7691
[IO] F1=0.7798 | Precision=0.7992 | Recall=0.7613
[BIOW] F1=0.7831 | Precision=0.8007 | Recall=0.7661
[BIOE] F1=0.7889 | Precision=0.8069 | Recall=0.7717
[BIOEW] F1=0.7843 | Precision=0.8018 | Recall=0.7675
[BIOW+] F1=0.7633 | Precision=0.7790 | Recall=0.7482

IDIOMA NEERLANDÈS
[BIO] F1=0.7719 | Precision=0.7942 | Recall=0.7508
[IO] F1=0.7579 | Precision=0.7851 | Recall=0.7325
[BIOW] F1=0.7678 | Precision=0.7905 | Recall=0.7464
[BIOE] F1=0.7669 | Precision=0.7871 | Recall=0.7477
[BIOEW] F1=0.7695 | Precision=0.7918 | Recall=0.7485
[BIOW+] F1=0.7624 | Precision=0.7859 | Recall=0.7403


# Prediccions (Cru)

In [50]:
mapeig_nl = {
    "NOUN": "N", "PROPN": "N", "VERB": "V", "AUX": "V",
    "ADJ": "Adj", "ADV": "Adv", "ADP": "Prep", "PRON": "Pron",
    "DET": "Art", "CCONJ": "Conj", "SCONJ": "Conj", "NUM": "Num",
    "PUNCT": "Punc", "SYM": "Punc", "INTJ": "Int"
}

In [51]:
import stanza

def preparar_text_real(dades: list, mapeig_neer: dict, idioma: str = "es") -> list:
    """    
    Entrada:
        dades:     llista de strings (text cru).
        idioma:    "es" per espanyol, "nl" per neerlandès.
    Sortida:
        x_pred:    Llista d'oracions amb format [(Token, POS, Lema), ...]
    """
    
    nlp = stanza.Pipeline(lang=idioma, processors='tokenize,pos,lemma', use_gpu=False, verbose=False)
    x_pred = []

    for text_oracio in dades:
        doc = nlp(text_oracio)
        oracio_x = []
        
        for oracio in doc.sentences:
            for paraula in oracio.words:
                lema = paraula.lemma
                
                if idioma == "es":
                    xpos_brut = paraula.xpos.upper() if paraula.xpos else paraula.upos.upper()
                    if xpos_brut.startswith("V"):
                        postag_oficial = xpos_brut[:3]
                    elif xpos_brut.startswith(("W", "Z", "I")):
                        postag_oficial = xpos_brut[:1]
                    elif xpos_brut.startswith("F"):
                        postag_oficial = xpos_brut.capitalize()
                    else:
                        postag_oficial = xpos_brut[:2]
                else:
                    postag_oficial = mapeig_neer.get(paraula.upos, paraula.upos)
                    
                oracio_x.append((paraula.text, postag_oficial, lema))   
                
        x_pred.append(oracio_x)
        
    return x_pred

In [52]:
# Comparativa alineada de postags entre el corpus y Stanza para español
oracio_conll_esp = esp_train[4]
paraules_corpus = [token[0] for token in oracio_conll_esp]
text_esp = " ".join(paraules_corpus)

x_pred_esp = preparar_text_real([text_esp], None, idioma="es")
oracio_stanza = x_pred_esp[0]
tokens_stanza = [t[0] for t in oracio_stanza]
postags_stanza = [t[1] for t in oracio_stanza]
mapa_stanza = {tok: postag for tok, postag in zip(tokens_stanza, postags_stanza)}

tabla = []

for token, postag_conll in zip(paraules_corpus, [t[1] for t in oracio_conll_esp]):
    postag_stanza = mapa_stanza.get(token, '-')
    tabla.append({'TOKEN': token, 'POSTAG (CoNLL)': postag_conll, 'POSTAG (Stanza)': postag_stanza})

df_comparativa = pd.DataFrame(tabla)
print(df_comparativa.to_string(index=False))

      TOKEN POSTAG (CoNLL) POSTAG (Stanza)
       Esta             DD              DD
     página             NC              NC
        web             AQ              NC
      lleva            VMI             VMI
         un             DI              DI
        mes             NC              NC
         de             SP              SP
 existencia             NC              NC
          ,             Fc              Fc
     tiempo             NC              NC
         en             SP              SP
         el             DA              DA
        que             PR              PR
         ha            VAI             VAI
       sido            VSP             VSP
   visitada            VMP             VMP
         en             SP              SP
        más             RG              RG
         de             SP              SP
         un             DI              DI
     millón             NC              NC
         de             SP              SP
  ocasiones

In [53]:
# Comparativa de codificación de postags entre el corpus y Stanza para neerlandés
oracio_conll_nl = ned_train[3]
paraules_originals_nl = [token[0] for token in oracio_conll_nl]
text_nl = " ".join(paraules_originals_nl)

x_pred_nl = preparar_text_real([text_nl], mapeig_nl, idioma="nl")
oracio_stanza = x_pred_nl[0]

tabla = []
for i in range(len(oracio_conll_nl)):
    token = oracio_conll_nl[i][0]
    postag_conll = oracio_conll_nl[i][1]
    postag_stanza = oracio_stanza[i][1] if i < len(oracio_stanza) else "-"
    tabla.append({'TOKEN': token, 'POSTAG (CoNLL)': postag_conll, 'POSTAG (Stanza)': postag_stanza})

df_comparativa = pd.DataFrame(tabla)
print(df_comparativa.to_string(index=False))

        TOKEN POSTAG (CoNLL) POSTAG (Stanza)
      Vandaag            Adv             Adv
           is              V               V
     Floralux              N               N
          dus            Adv             Adv
          met           Prep            Prep
         alle           Pron             Art
 vergunningen              N               N
           in           Prep            Prep
         orde              N               N
            ,           Punc            Punc
         maar           Conj            Conj
          het            Art             Art
          BPA              N               N
      waarmee            Adv             Adv
          die           Pron            Pron
       konden              V               V
    verkregen              V               V
       worden              V               V
            ,           Punc            Punc
          was              V               V
    omstreden            Adj               V
        om

In [54]:
def predir_text_cru(textos: list, tagger, idioma: str, mapeig_neer: dict = None) -> list:
    """
    Fa prediccions d'entitats sobre text cru utilitzant un model CRF prèviament entrenat.
    
    Entrada:
        textos: Llista de cadenes de text (strings) a predir.
        tagger: L'objecte CRFTagger ja entrenat.
        idioma: "es" o "nl".
        mapeig_neer: Diccionari de traducció POS per al neerlandès (pot ometre's si és "es").
    """
    if mapeig_neer is None:
        mapeig_neer = {}
        
    # Preparar el text cru amb els postags correctes (han de cuadrar amb el del corpus)
    x_pred_cru = preparar_text_real(textos, mapeig_neer, idioma)
    
    # PREDICCIÓ instantània utilitzant el tagger passat per paràmetre
    prediccions_cru = tagger.tag_sents(x_pred_cru)
    
    # VISUALITZACIÓ
    print(f"\n=== RESULTATS DE LA PREDICCIÓ EN TEXT CRU ===")
    for i, oracio_predita in enumerate(prediccions_cru):
        print(f"\n--- Oració {i+1} ---")
        print(f"TEXT ORIGINAL: {textos[i]}")
        
        entitats_trobades = False
        for token, etiqueta in oracio_predita:
            # Filtrem 'O' i 'O-' per mostrar només les entitats reals
            if etiqueta != 'O' and not etiqueta.startswith('O-'):
                print(f"[{etiqueta:^7}] -> {token}")
                entitats_trobades = True
                
        if not entitats_trobades:
            print("No s'han detectat entitats en aquesta oració.")
            
    return prediccions_cru

In [28]:
# =========================================================
# CONFIGURACIÓ I ENTRENAMENT - ESPANYOL
# =========================================================
corpus_transf_esp = transformar_codificacio(esp_train, "BIOE")
    
# Obtenir els lemes i preparar el corpus
x_train_final_esp = preparar_dades(corpus_transf_esp, idioma="es", entrenament=True)

# Instanciar l'extractor amb els paràmetres guanyadors
extractor_final_esp = FeatFunc_Class(
    gasetes_freq=gasetes_freq_es, 
    gasetes_simples=gasetes_simp, 
    **param_esp)

# Entrenar el model
tagger_final_esp = CRFTagger(feature_func=extractor_final_esp)
tagger_final_esp.train(x_train_final_esp, "model_final_bioe_esp.crf.tagger")

In [55]:
x_test_esp, y_test_esp = preparar_dades(esp_test,  idioma="es", entrenament=False)

y_pred_esp = tagger_final_esp.tag_sents(x_test_esp)
y_pred_etiquetes_esp = [[ner for _, ner in oracio] for oracio in y_pred_esp]
f1_p_esp, precision_esp, recall_esp = f1_score_parcial(y_test_esp, y_pred_etiquetes_esp, "BIOE")

print(f"Rendiment final del model d'Espanyol amb el conjunt test:")
print(f"   Configuarció: BIOE")
print(f"   F1: {f1_p_esp}")
print(f"   Precision: {precision_esp}")
print(f"   Recall: {recall_esp}")

Rendiment final del model d'Espanyol amb el conjunt test:
   Configuarció: BIOE
   F1: 0.8129363157144892
   Precision: 0.8243282288355966
   Recall: 0.8018549747048904


In [56]:
# =========================================================
# 2. PREDICCIÓ - ESPANYOL
# =========================================================
textos_prova = [
    "Mark Petersen está trabajando en Google desde sus oficinas en Barcelona.",
    "El presidente Pedro Sánchez se reunió con la ONU en Nueva York la semana pasada.",
    "Sílvia Orriols ha convertido el hemiciclo del Parlament en su plató más eficaz.",
    "No participa en las comisiones y desaparece cuando otros grupos interpelan al Govern." "Orriols exprime sus cinco minutos de confrontación con el presidente de la Generalitat y es capaz de provocar el caos con una frase",
    "'Aprovecho los segundos que me quedan para darle las gracias por los presupuestos y por desarticular al PSC en Ripoll' lanzó a Salvador Illa el miércoles.",
    "El tono desafiante de Orriols es proporcional al auge que dan las encuestas a Aliança Catalana.", 
    "La Barcelona que por la mañana recibió a Lula da Silva (Brasil) y a Gustavo Petro (Colombia) y que por la noche cantó la llegada de la presidenta de México, Claudia Sheinbaum, la Barcelona del tenis en el Godó y también la de los desfiles en la 080, todo en la misma semana, se ha entregado sin dudarlo a Rosalía.",
    "Pese a todos los rumores, no le tocó el turno ni a Shakira, ni a Belén Esteban, ni siquiera a Pedro Sánchez, presidente del Gobierno de España, que hacía noche en Barcelona.",
    "La llamada a confesarse fue Bad Gyal.",
    "La alarma la ha encendido Aonan Guan y los investigadores de Johns Hopkins Zhengyu Liu y Gavin Zhong tras demostrar ataques contra tres agentes desplegados en la mencionada plataforma: Claude Code Security Review, de Anthropic, Gemini CLI Action, de Google, y GitHub Copilot Agent, una herramienta de GitHub bajo Microsoft.",
    "El petróleo proviene no solo de Irán sino también de otros Estados del Golfo, tales como Irak, Kuwait, Qatar, Arabia Saudita y los EAU.",
    "Cuando comparamos la deuda en proporción al PIB, la de Costa Rica con el Fondo Monetario no es tan pesada como la de Argentina o Ecuador, que son dos países que han estado en situaciones fiscales sumamente complicadas, explica a BBC Mundo Luis Mesalles, economista y socio de la consultora Ecoanálisis.",
    "Aunque se mantiene ligeramente por encima de la media de los países de la OCDE, Suecia tuvo peores resultados en alfabetización en 2022 que países como Reino Unido, Estados Unidos, Dinamarca y Finlandia.",
    "SMIC, el mayor productor chino de semiconductores, actualmente solo puede fabricar chips de 7 nm empleando la técnica de multiple patterning. Y TSMC, Intel y Samsung, que podrían producirlo, difícilmente lo harán en el contexto geopolítico actual debido a las exigencias de las sanciones a China de EEUU.",
    "Veremos cómo resuelve este desafío Dishan Technology. Por otro lado, Moore Threads ha desarrollado varias GPU para aplicaciones de IA que, sobre el papel, rivalizan con algunas de las soluciones avanzadas que han colocado en el mercado NVIDIA, AMD o Huawei."]

# Es crida la funció passant-li ÚNICAMENT el tagger ja entrenat i els textos
prediccions_finals = predir_text_cru(
    textos=textos_prova,
    tagger=tagger_final_esp,  # Li passem l'objecte directe
    idioma="es",
    mapeig_neer={} # en espanyol no cal diccionari de mapeig (es fa amb slicing directament)
)


=== RESULTATS DE LA PREDICCIÓ EN TEXT CRU ===

--- Oració 1 ---
TEXT ORIGINAL: Mark Petersen está trabajando en Google desde sus oficinas en Barcelona.
[ B-PER ] -> ('Mark', 'NP', 'mark')
[ E-PER ] -> ('Petersen', 'PR', 'Petersen')
[ B-LOC ] -> ('Google', 'NP', 'google')
[ B-LOC ] -> ('Barcelona', 'NP', 'Barcelona')

--- Oració 2 ---
TEXT ORIGINAL: El presidente Pedro Sánchez se reunió con la ONU en Nueva York la semana pasada.
[ B-PER ] -> ('Pedro', 'NP', 'pedro')
[ E-PER ] -> ('Sánchez', 'PR', 'Sánchez')
[ B-ORG ] -> ('ONU', 'NP', 'ONU')
[ B-LOC ] -> ('Nueva', 'NP', 'nueva')
[ E-LOC ] -> ('York', 'PR', 'York')

--- Oració 3 ---
TEXT ORIGINAL: Sílvia Orriols ha convertido el hemiciclo del Parlament en su plató más eficaz.
[ B-PER ] -> ('Sílvia', 'NP', 'Sílvia')
[ E-PER ] -> ('Orriols', 'PR', 'Orriols')
[ B-ORG ] -> ('Parlament', 'NP', 'Parlament')

--- Oració 4 ---
TEXT ORIGINAL: No participa en las comisiones y desaparece cuando otros grupos interpelan al Govern.Orriols exprime sus 

In [33]:
# =========================================================
# 1. CONFIGURACIÓ I ENTRENAMENT - NEERLANDÈS
# =========================================================
corpus_transf_nl = transformar_codificacio(ned_train, "BIO")
    
# Obtenir els lemes i preparar el corpus
x_train_final_nl = preparar_dades(corpus_transf_nl, idioma="nl", entrenament=True)

# Instanciar l'extractor amb els paràmetres guanyadors
extractor_final_nl = FeatFunc_Class(
    gasetes_freq=gasetes_freq_nl , 
    gasetes_simples=gasetes_simp, 
    **param_neer)

# Entrenar el model
tagger_final_nl = CRFTagger(feature_func=extractor_final_nl)
tagger_final_nl.train(x_train_final_nl, "model_final_bio_nl.crf.tagger")

In [57]:
x_test_nl, y_test_nl = preparar_dades(ned_test,  idioma="nl", entrenament=False)

y_pred_nl = tagger_final_nl.tag_sents(x_test_nl)
y_pred_etiquetes_nl = [[ner for _, ner in oracio] for oracio in y_pred_nl]
f1_p_nl, precision_nl, recall_nl = f1_score_parcial(y_test_nl, y_pred_etiquetes_nl, "BIO")

print(f"Rendiment final del model de Neerlandès amb el conjunt test:")
print(f"   Configuarció: BIO")
print(f"   F1: {f1_p_nl}")
print(f"   Precision: {precision_nl}")
print(f"   Recall: {recall_nl}")

Rendiment final del model de Neerlandès amb el conjunt test:
   Configuarció: BIO
   F1: 0.7869321879474164
   Precision: 0.807856761090326
   Recall: 0.767064196904339


In [58]:
# =========================================================
# 2. PREDICCIÓ - NEERLANDÈS
# =========================================================
textos_prova_nl = [
    "Verstappen was op het moment van de crash niet op het circuit.",
    "Zijn teamgenoot Lucas Auer was begonnen aan de race, die tot 21.30 uur zou duren.",
    "Op zondag staat er nog een race op het programma op de Nürburgring.", 
    "De rit van het Penn Station in Manhattan naar het stadion in East Rutherford (New Jersey) is ongeveer 14 kilometer lang en duurt zo'n 15 minuten.",
    "De gouverneur van New Jersey, Mikie Sherrill, heeft de internationale voetbalbond FIFA opgeroepen de transportkosten te vergoeden.", 
    "Een patrouille van de VN-vredesmacht in Libanon (UNIFIL) is zaterdagochtend aangevallen door een groepering.",
    "Volgens de Franse president Emmanuel Macron zit Hezbollah achter de aanslag, maar die militie ontkent.",
    "Paus Leo XIV zegt zaterdag dat zijn tirannenspeech twee weken geleden al is voorbereid.",
    "'Ruim vóór Trump reageerde op mij en de vredesboodschap die ik verkondig', vervolgt de paus, die ook van Amerikaanse afkomst is.",
    "Overigens brak de oorlog van de VS en Israël tegen Iran eind februari al uit en kan het op die manier alsnog als inspiratie hebben gediend voor de speech.",
    "Oprichter Dario Amodei van AI-bedrijf Anthropic heeft op het Witte Huis de ontwikkeling van super-AI Mythos besproken.",
    "De ontmoeting is opmerkelijk, omdat president Trump Anthropic juist in de ban heeft gedaan omdat het bedrijf wil vasthouden aan strenge ethische regels.",
    "Zelfrijdende Tesla's mogen de weg op in Nederland.",
    "De RDW heeft goedkeuring gegeven voor het rijhulpsysteem Full Self Driving (Supervised).",
    "Nederland is het eerste land in Europa waar Tesla dit systeem mag gebruiken.",
    "Het technologiebedrijf achter ChatGPT, OpenAI, stopt met video-generatorapp Sora.",
    "The Walt Disney Company groots aan te gaan samenwerken met OpenAI door een miljard dollar te investeren en 200 karakters beschikbaar te stellen in Sora."]

# Es crida la funció passant-li ÚNICAMENT el tagger ja entrenat i els textos
prediccions_finals = predir_text_cru(
    textos=textos_prova_nl,
    tagger=tagger_final_nl,  # Li passem l'objecte directe
    idioma="nl",
    mapeig_neer=mapeig_nl
)


=== RESULTATS DE LA PREDICCIÓ EN TEXT CRU ===

--- Oració 1 ---
TEXT ORIGINAL: Verstappen was op het moment van de crash niet op het circuit.
No s'han detectat entitats en aquesta oració.

--- Oració 2 ---
TEXT ORIGINAL: Zijn teamgenoot Lucas Auer was begonnen aan de race, die tot 21.30 uur zou duren.
[ B-PER ] -> ('Lucas', 'N', 'Lucas')
[ I-PER ] -> ('Auer', 'N', 'Auer')

--- Oració 3 ---
TEXT ORIGINAL: Op zondag staat er nog een race op het programma op de Nürburgring.
[ B-LOC ] -> ('Nürburgring', 'N', 'Nürburgring')

--- Oració 4 ---
TEXT ORIGINAL: De rit van het Penn Station in Manhattan naar het stadion in East Rutherford (New Jersey) is ongeveer 14 kilometer lang en duurt zo'n 15 minuten.
[ B-ORG ] -> ('Penn', 'N', 'Penn')
[ I-ORG ] -> ('Station', 'N', 'Station')
[ B-LOC ] -> ('Manhattan', 'N', 'Manhattan')
[ B-LOC ] -> ('East', 'N', 'East')
[ I-LOC ] -> ('Rutherford', 'N', 'Rutherford')
[B-MISC ] -> ('New', 'N', 'New')
[I-MISC ] -> ('Jersey', 'N', 'Jersey')

--- Oració 5 ---
TE